In [ ]:
# ============================================================
# NOEMA RSI — CONTINUAR USANDO LOS ARCHIVOS YA SUBIDOS A COLAB
# ============================================================

GENERATIONS = 100
PATIENCE = 50
DEVICE = "cuda"   # "cuda" si quieres GPU y está disponible

from pathlib import Path
import subprocess
import sys
import zipfile
import tempfile
import shutil
from google.colab import files


CONTENT = Path("/content")

# ------------------------------------------------------------
# 1. ENCONTRAR AUTOMÁTICAMENTE EL ZIP DEL PROYECTO
# ------------------------------------------------------------

project_candidates = [
    p for p in CONTENT.glob("*.zip")
    if (
        "asi" in p.name.lower()
        or "agi-code-agi" in p.name.lower()
    )
    and "resultado" not in p.name.lower()
]

if not project_candidates:
    raise FileNotFoundError(
        "No encontré ASI.zip o AGI-Code-AGI.zip en /content"
    )

# Usa el ZIP de proyecto modificado más recientemente
PROJECT_ZIP = max(
    project_candidates,
    key=lambda p: p.stat().st_mtime
)

print("Proyecto seleccionado:")
print(" ", PROJECT_ZIP)


# ------------------------------------------------------------
# 2. BUSCAR CHECKPOINT
# ------------------------------------------------------------

direct_checkpoint = CONTENT / "checkpoint.pt"

if direct_checkpoint.exists():

    CHECKPOINT = direct_checkpoint

    print("\nCheckpoint directo encontrado:")
    print(" ", CHECKPOINT)

else:

    # Si no existe checkpoint.pt suelto,
    # buscar el ZIP de resultados más reciente.

    result_zips = list(CONTENT.glob("*resultado*.zip"))

    if not result_zips:
        raise FileNotFoundError(
            "No encontré /content/checkpoint.pt "
            "ni ningún NOEMA_resultados*.zip"
        )

    RESULTS_ZIP = max(
        result_zips,
        key=lambda p: p.stat().st_mtime
    )

    print("\nExtrayendo checkpoint desde:")
    print(" ", RESULTS_ZIP)

    checkpoint_temp = Path(
        tempfile.mkdtemp(prefix="checkpoint_")
    )

    with zipfile.ZipFile(RESULTS_ZIP) as z:
        z.extractall(checkpoint_temp)

    checkpoints = list(
        checkpoint_temp.rglob("checkpoint.pt")
    )

    if not checkpoints:
        raise FileNotFoundError(
            f"{RESULTS_ZIP.name} no contiene checkpoint.pt"
        )

    CHECKPOINT = max(
        checkpoints,
        key=lambda p: p.stat().st_mtime
    )


print("\nCHECKPOINT QUE SE VA A USAR:")
print(" ", CHECKPOINT)

print(
    f"Tamaño: "
    f"{CHECKPOINT.stat().st_size / 1024**2:.2f} MB"
)


# ------------------------------------------------------------
# 3. EXTRAER PROYECTO
# ------------------------------------------------------------

work = Path(tempfile.mkdtemp(prefix="noema_continue_"))
project_root = work / "project"
project_root.mkdir()


def safe_extract(zip_path, destination):

    destination = destination.resolve()

    with zipfile.ZipFile(zip_path) as archive:

        for item in archive.infolist():

            target = (
                destination / item.filename
            ).resolve()

            if not target.is_relative_to(destination):
                raise ValueError(
                    f"Ruta peligrosa: {item.filename}"
                )

            # No permitir symlinks
            mode = item.external_attr >> 16

            if mode & 0o170000 == 0o120000:
                raise ValueError(
                    f"Symlink no permitido: {item.filename}"
                )

        archive.extractall(destination)


safe_extract(PROJECT_ZIP, project_root)


# ------------------------------------------------------------
# 4. ENCONTRAR run_rsi.py
# ------------------------------------------------------------

run_rsi = next(
    project_root.rglob("run_rsi.py"),
    None
)

if run_rsi is None:
    raise FileNotFoundError(
        "No encontré run_rsi.py dentro del proyecto"
    )

project = run_rsi.parent

print("\nrun_rsi.py:")
print(" ", run_rsi)


# ------------------------------------------------------------
# 5. INSTALAR PROYECTO
# ------------------------------------------------------------

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-e",
    str(project)
])


# ------------------------------------------------------------
# 6. CONTINUAR RSI DESDE EL CHECKPOINT
# ------------------------------------------------------------

output = work / "resultados_continuados"

command = [
    sys.executable,
    str(run_rsi),

    "--resume",
    str(CHECKPOINT),

    "--generations",
    str(GENERATIONS),

    "--patience",
    str(PATIENCE),

    "--device",
    DEVICE,

    "--output",
    str(output),
]

print()
print("=" * 70)
print("             CONTINUANDO NOEMA RSI")
print("=" * 70)
print("Proyecto:    ", PROJECT_ZIP.name)
print("Checkpoint:  ", CHECKPOINT)
print("Generaciones:", GENERATIONS)
print("Patience:    ", PATIENCE)
print("Device:      ", DEVICE)
print("=" * 70)
print()

subprocess.check_call(
    command,
    cwd=project
)


# ------------------------------------------------------------
# 7. MOSTRAR REPORTE
# ------------------------------------------------------------

report = output / "report.json"

if report.exists():

    print()
    print("=" * 70)
    print("REPORTE FINAL")
    print("=" * 70)

    print(
        report.read_text(
            encoding="utf-8"
        )[:6000]
    )


# ------------------------------------------------------------
# 8. GUARDAR NUEVO CHECKPOINT TAMBIÉN EN /content
# ------------------------------------------------------------

new_checkpoint = output / "checkpoint.pt"

if new_checkpoint.exists():

    persistent_checkpoint = (
        CONTENT / "checkpoint_NUEVO.pt"
    )

    shutil.copy2(
        new_checkpoint,
        persistent_checkpoint
    )

    print()
    print("Nuevo checkpoint guardado también en:")
    print(" ", persistent_checkpoint)


# ------------------------------------------------------------
# 9. CREAR ZIP FINAL
# ------------------------------------------------------------

archive_path = shutil.make_archive(
    str(CONTENT / "NOEMA_resultados_CONTINUADOS"),
    "zip",
    output
)

print()
print("=" * 70)
print("TERMINADO")
print("=" * 70)
print("Resultados:")
print(archive_path)

files.download(archive_path)

Proyecto seleccionado:
  /content/ASI (1).zip

Checkpoint directo encontrado:
  /content/checkpoint.pt

CHECKPOINT QUE SE VA A USAR:
  /content/checkpoint.pt
Tamaño: 4.87 MB

run_rsi.py:
  /tmp/noema_continue_laf1zsvo/project/AGI-Code-AGI/run_rsi.py

             CONTINUANDO NOEMA RSI
Proyecto:     ASI (1).zip
Checkpoint:   /content/checkpoint.pt
Generaciones: 100
Patience:     50
Device:       cuda

